# Full baseline10XE V3 — local FP16 cache, T4×2

InternetON, Accelerator **GPU T4×2**, Save & Run All. Attach only `ducanh2403/visual-cache` and `vuthetam/mscoco-2014`; same paths below. No API token/bank/prompt/old checkpoint. V1 stopped at0mainupdates; this V3 starts fresh seed42, same architecture/10XE/allTRAINcaptionevents/global32/schedule/VALID policy.

V3 copies all113287TRAIN+5000VALID patches sequentially to **/tmp local float16 memmap (~33.17GiB)** before training. No CLIP recomputation, no fullRAM preload, no TESTfeature reads. Need38GiB /tmp plus4GiB Working. Preserve all196×768patchvalues; exact source/destination byte checks,256smoke rawrowSHA, mapping coverage and file seals. Stage includes sequential SHA verification; progress/heartbeat may continue while sealing. V2 timed out halfway during its1800s preparation; V3 starts fresh, preparation cap5400s(90min); parent10.5h STOPcap unchanged, not ETA/billing promise.

Fulllocalcorpus exceeds prior RAM31GiB, unlike smallCPUprobe. Actual fullIO/DDP/math/beam preflight mandatory, warmupupdates discarded, only train if measured projection fits remaining cap. Periodic budget gate remains; no automatic extension. Fixed10epochs/177110updates, global32; eachVALID5000 beam5/NLL/CIDEr. Best=maxVALIDCIDEr, keepbothbest/latest withAdam/RNG/cursor. Finalbest/end quality uses savedcaptions: BLEU1/4,ROUGE-L,CHAIRs/i/recall. METEOR/SPICE/TEST later; noSCST/reranking/prompt/bank.

Send `clip_patch_baseline_full_xe10_v3_review.zip`. Preserve outputfolder/checkpoint.bin/best.bin once created; /tmp is disposable so resume requires exactV3 metadata/cache restaging and preservedcheckpointoutput. No resume fromV1/V2differentcode. Author verified onlyNumPy/mock/AST; realfullIO/GPU/scorers remain user-run.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile
from pathlib import Path
REVISION='9aaddf5694f72373b8ab2482fafe7d7992abbe5e'
MAPPING={'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/full_runtime.py': 'full_runtime.py', 'experiments/clip_patch_baseline/run_full.py': 'run_full.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/score_full.py': 'score_full.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/test_full_data.py': 'test_full_data.py', 'experiments/clip_patch_baseline/make_full_notebook.py': 'make_full_notebook.py', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'FULL_XE10_PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/clip_patch_baseline/local_stage.py': 'local_stage.py', 'experiments/clip_patch_baseline/test_local_stage.py': 'test_local_stage.py', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': 'LOCAL_STAGE_V3_PROTOCOL.md'}
HASHES={'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/full_runtime.py': 'f00cee0843d71679cc892ce18f4a731ae759d6918ee9e01d14f1c259b233bdb7', 'experiments/clip_patch_baseline/run_full.py': '276e2936f47b38b33ce094efb2a51c79af73f1adcf92b644da42c1ce567f06b0', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/score_full.py': '7af818128c62e30a176a16c0944e6346389b52b3c9c758d98e840d8eb1ceedda', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/test_full_data.py': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431', 'experiments/clip_patch_baseline/make_full_notebook.py': '132ce1f84d6e8bb0822082740a068028e6307e527dff0bc44c2ac4f90c39a7df', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/clip_patch_baseline/local_stage.py': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b', 'experiments/clip_patch_baseline/test_local_stage.py': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
RESUME_ROOT='' # Optional preserved full-run output from THIS exact recipe. Blank starts fresh.
WALL_HOURS=10.5 # Parent stop cap, not ETA/billed quota. Do not auto-extend.
OUTPUT=Path('/kaggle/working/clip_patch_baseline_full_xe10_v3')
for p in (VISUAL_CACHE,COCO_JSON):
    if not p.exists():raise FileNotFoundError('Attach/correct inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh runtime/output path')
if shutil.disk_usage('/tmp').free<38*1024**3 or shutil.disk_usage('/kaggle/working').free<4*1024**3:raise RuntimeError('Need38GiB temporary scratch and4GiB Working; sequential local stage required')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-full10-',dir='/tmp'));DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair'
cfg=dict(feature_storage='local_fp16_memmap_v2',prepare_seconds=5400,output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600,resume_root=RESUME_ROOT,chair=str(CHAIR))
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned asset: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Full patch XE active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    for name,destination in MAPPING.items():download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,HASHES[name],SOURCE/destination)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_local_stage.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab');assert nltk.word_tokenize('two dogs')==['two','dogs']"],300)
    base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
    for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d','chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():download(base+name,digest,CHAIR/name)
    java_archive=SOURCE/'java8.tar.gz'
    download('https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz','6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e',java_archive)
    with tarfile.open(java_archive) as z:z.extractall(SOURCE/'jre',filter='data')
    java=list((SOURCE/'jre').rglob('bin/java'))
    if len(java)!=1:raise ValueError('Missing pinned Java8')
    os.environ['PATH']=str(java[0].parent)+os.pathsep+os.environ['PATH']
    run([java[0],'-version'],60)
    run([sys.executable,'-u',SOURCE/'score_full.py','probe','--config',CONFIG],180)
    run([sys.executable,'-u',SOURCE/'run_full.py','prepare','--config',CONFIG],5400)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    cfg['worker_seconds']=min(1200,PARENT_DEADLINE-time.monotonic()-1800);write_config()
    if cfg['worker_seconds']<180:raise TimeoutError('Insufficient budget for real IO preflight')
    run([*launch,SOURCE/'run_full.py','preflight','--config',CONFIG],cfg['worker_seconds']+30)
    cfg['worker_seconds']=PARENT_DEADLINE-time.monotonic()-1200;write_config()
    if cfg['worker_seconds']<1800:raise TimeoutError('Insufficient full-run budget after setup')
    run([*launch,SOURCE/'run_full.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    run([sys.executable,'-u',SOURCE/'score_full.py','score','--config',CONFIG],1200)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='CLIP_PATCH_FULL_XE10_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,epochs_requested=10,test_used=False,SCST=False,reranking=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('LATEST:',OUTPUT/'checkpoint.bin','BEST VALID:',OUTPUT/'best.bin',flush=True)
    print('FULL10 COMPLETE; no TEST or prompt/bank yet.' if complete else 'INCOMPLETE; preserve weights/logs; no automatic restart or partial-model ranking.',flush=True)
